# 02 — Exploratory Data Analysis (EDA)
Understand the dataset before modelling.  
Key questions: distribution of power, correlations, circuit complexity spread.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

all_data = pd.read_csv('data/processed/all_circuits.csv')
train    = pd.read_csv('data/processed/train.csv')
test     = pd.read_csv('data/processed/test.csv')

FEATURES = ['gate', 'and', 'inv', 'nor', 'nand', 'or', 'dff', 'in', 'out']
FEATURE_LABELS = ['Total gates', 'AND', 'Inverters', 'NOR', 'NAND',
                  'OR', 'Flip-flops', 'Inputs', 'Outputs']
os.makedirs('results/figures', exist_ok=True)
print("✓ Data loaded — n =", len(all_data), "circuits")


In [ ]:
# ── Complexity grouping (used throughout paper) ──
def complexity_group(g):
    if g < 200:   return 'Small (<200)'
    elif g < 500: return 'Medium (200–500)'
    else:         return 'Large (>500)'

all_data['complexity'] = all_data['gate'].apply(complexity_group)
print(all_data[['circuit','gate','power','complexity']].to_string(index=False))


In [ ]:
# ── Figure: Power distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(all_data['power'], bins=15, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Power (W)')
axes[0].set_ylabel('Count')
axes[0].set_title('Power distribution (linear scale)')
axes[0].spines[['top','right']].set_visible(False)

axes[1].hist(np.log10(all_data['power']), bins=15, color='#DD8452', edgecolor='white')
axes[1].set_xlabel('log₁₀(Power)')
axes[1].set_ylabel('Count')
axes[1].set_title('Power distribution (log scale)')
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('ISCAS\'89 — Power distribution (n=25)', fontsize=12)
plt.tight_layout()
plt.savefig('results/figures/eda_power_dist.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure: Feature correlation heatmap ──
fig, ax = plt.subplots(figsize=(10, 8))
corr = all_data[FEATURES + ['power']].corr()
labels = FEATURE_LABELS + ['Power']
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=ax, linewidths=0.5,
            xticklabels=labels, yticklabels=labels)
ax.set_title("Feature Correlation Matrix — ISCAS\'89", fontsize=12)
plt.tight_layout()
plt.savefig('results/figures/eda_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

# Print correlation with power (most important for paper)
print("\nCorrelation with power (sorted):")
print(corr['power'].drop('power').sort_values(ascending=False).round(4))


In [ ]:
# ── Figure: Feature distributions by complexity group ──
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
colors = {'Small (<200)':'#55A868', 'Medium (200–500)':'#DD8452', 'Large (>500)':'#C44E52'}

for i, feat in enumerate(FEATURES):
    for grp, grp_df in all_data.groupby('complexity'):
        axes[i].scatter(grp_df[feat], grp_df['power'],
                        label=grp, color=colors[grp], s=60, alpha=0.8)
    axes[i].set_xlabel(FEATURE_LABELS[i], fontsize=9)
    axes[i].set_ylabel('Power (W)', fontsize=9)
    axes[i].spines[['top','right']].set_visible(False)

axes[9].axis('off')
handles = [plt.Line2D([0],[0], marker='o', color='w',
           markerfacecolor=c, markersize=9, label=l)
           for l, c in colors.items()]
axes[9].legend(handles=handles, title='Complexity', loc='center', fontsize=10)

plt.suptitle('Feature vs Power by Circuit Complexity', fontsize=13)
plt.tight_layout()
plt.savefig('results/figures/eda_feature_scatter.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure: Gate count by circuit (for paper) ──
all_sorted = all_data.sort_values('gate')
color_map  = all_sorted['complexity'].map(colors)

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(all_sorted['circuit'], all_sorted['gate'],
              color=color_map, edgecolor='white', linewidth=0.6)
ax.set_ylabel('Gate count')
ax.set_title('ISCAS\'89 Circuits — Gate count (complexity groups)')
ax.tick_params(axis='x', rotation=45)
handles = [plt.Rectangle((0,0),1,1, color=c, label=l) for l, c in colors.items()]
ax.legend(handles=handles)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/figures/eda_gate_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


---
**Key EDA findings to mention in paper:**
- Power spans 3 orders of magnitude → log-scale analysis justified
- `gate` count most correlated with power (note value from output above)
- Clear complexity tiers: small / medium / large
- Dataset is small (n=25) → LOOCV used for robust evaluation

**Next:** Run `03_preprocessing.ipynb`